In [ ]:
# Import libraries:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timedelta
from pathlib import Path
import os
import glob

import numpy as np
import pandas as pd
import xarray as xr
# import s3fs
import pyart
from pyproj import Proj, Geod

import matplotlib.pyplot as plt
from matplotlib.pyplot import cm as cmaps
import cartopy.crs as ccrs

import pyPIPS.utils as utils
import pyPIPS.radarmodule as radar

# Import tobac itself:
import tobac
import tobac.utils

print("using tobac version", str(tobac.__version__))
%matplotlib inline

In [ ]:
# Define some functions for reading CM1 and COMMAS output

def read_CM1(ncdir, prefix='cm1out', one_time_per_file=True):
    """Reads one or more CM1 netCDF files into an xarray Dataset

    Parameters
    ----------
    ncdir : str
        Directory where the CM1 netCDF file(s) reside
    prefix : str, optional
        CM1 file name prefix, by default 'cm1out'
    one_time_per_file : bool, optional
        Whether each output time is contained in a separate file, by default True

    Returns
    -------
    xarray.Dataset
        An xarray Dataset containing the data from the netCDF file(s)
    """

    # Construct the file path
    if one_time_per_file:
        cm1_filename = prefix + '_[0-9]*.nc'
    else:
        cm1_filename = prefix + '.nc'
    cm1_path = os.path.join(ncdir, cm1_filename)

    # Open up the netCDF file(s) using xarray
    # NOTE: passing decode_cf=False because encoding not present in CM1 output files
    if one_time_per_file:
        cm1_ds = xr.open_mfdataset(cm1_path, decode_cf=False)
    else:
        cm1_ds = xr.open_dataset(cm1_path, decode_cf=False)

    return cm1_ds

def read_COMMAS(ncdir, prefix='commasout', member=0, one_time_per_file=True):
    """Reads one or more COMMAS netCDF files into an xarray Dataset

    Parameters
    ----------
    ncdir : str
        Directory where the COMMAS netCDF file(s) reside
    prefix : str, optional
        COMMAS file name prefix, by default 'commasout'
    member : int, optional
        COMMAS ensemble member number, by default 0
    one_time_per_file : bool, optional
        Whether each output time is contained in a separate file, by default True

    Returns
    -------
    xarray.Dataset
        An xarray Dataset containing the data from the netCDF file(s)
    """

    # Construct the file path
    if one_time_per_file:
        commas_filename = prefix + '.[0-9]*.nc'
    else:
        commas_filename = prefix + f'.{member:03d}.000.nc'
    commas_path = os.path.join(ncdir, commas_filename)

    # Open up the netCDF file(s) using xarray
    # NOTE: passing decode_cf=False because encoding not present in COMMAS output files
    if one_time_per_file:
        commas_ds = xr.open_mfdataset(commas_path, decode_cf=False)
    else:
        commas_ds = xr.open_dataset(commas_path, decode_cf=False)

    return commas_ds


def read_sweeps(radname, radardir, radstarttime, radstoptime, fieldnames, el_req):
    """Reads sweeps from CFRadial files for a case"""
    radpathlist = glob.glob(radardir + f'/*{radname}*nc')

    # Now read in all the sweeps between radstarttime and radstoptime closest to the requested
    # elevation angle
    radstarttimedt = datetime.strptime(radstarttime, '%Y%m%d%H%M%S')
    radstoptimedt = datetime.strptime(radstoptime, '%Y%m%d%H%M%S')

    # outfieldnameslist = []
    radarsweeplist = []
    sweeptimelist = []

    for radpath in radpathlist:
        sweeptime1 = radar._getsweeptime(radpath)

        if radstarttimedt <= sweeptime1 and sweeptime1 <= radstoptimedt:
            # Note, the following may extract more than one sweep if there are multiple sweeps
            # with the same elevation angle in the file (i.e. as in SAILS mode)
            radarsweep1 = radar.readCFRadial_pyART(el_req, radpath, sweeptime1,
                                                   fieldnames, compute_kdp=False)
            for radarsweep in radarsweep1:
                # A slight gotcha here... The sweep number stored in the extracted radarsweep
                # is the sweep number in the original file. So if there are multiple sweeps with
                # the same elevation angle in the original file, like with SAILS mode, then all the
                # sweeps after the first one will inherit the number they had in the original file,
                # instead of setting it to zero in the extracted radarsweep. This causes problems
                # later when, e.g., trying to use grid_ppi_sweeps. So we need to reset the
                # sweep number to zero for all the extracted sweeps.
                radarsweep.sweep_number['data'] = np.zeros_like(radarsweep.sweep_number['data'])
                radarsweeplist.append(radarsweep)
                # Extract time from start of each sweep
                sweep_time = pyart.graph.common.generate_radar_time_sweep(radarsweep, 0)
                sweeptimelist.append(sweep_time)

    # Sort the lists by increasing time since glob doesn't sort in any particular order
    sorted_sweeptimelist = sorted(sweeptimelist)
    sorted_radarsweeplist = [x for _, x in sorted(zip(sweeptimelist, radarsweeplist),
                                                  key=lambda pair: pair[0])]

    return sorted_sweeptimelist, sorted_radarsweeplist



In [ ]:
case_config_path = '/Users/dawson29/Projects/pyPIPS/configs/ICECHIP_IOP12_2025_10s.py'
input_tag = None
dealias_vel = False
el_req_cl = None

# Dynamically import the case configuration file
utils.log("Case config file is {}".format(case_config_path))
config = utils.import_all_from(case_config_path)
try:
    config = utils.import_all_from(case_config_path)
    utils.log("Successfully imported case configuration parameters!")
except Exception:
    utils.fatal(
        "Unable to import case configuration parameters! Aborting!")

# Extract needed lists and variables from PIPS_IO_dict configuration dictionary
dataset_name = config.PIPS_IO_dict.get('dataset_name', None)
deployment_names = config.PIPS_IO_dict.get('deployment_names', None)
PIPS_dir = config.PIPS_IO_dict.get('PIPS_dir', None)
plot_dir = config.PIPS_IO_dict.get('plot_dir', None)
PIPS_types = config.PIPS_IO_dict.get('PIPS_types', None)
PIPS_names = config.PIPS_IO_dict.get('PIPS_names', None)
PIPS_filenames = config.PIPS_IO_dict.get('PIPS_filenames', None)
parsivel_combined_filenames = config.PIPS_IO_dict['PIPS_filenames_nc']
start_times = config.PIPS_IO_dict.get('start_times', [None] * len(PIPS_names))
end_times = config.PIPS_IO_dict.get('end_times', [None] * len(PIPS_names))
geo_locs = config.PIPS_IO_dict.get('geo_locs', [None] * len(PIPS_names))
requested_interval = config.PIPS_IO_dict.get('requested_interval', 10.)

# Extract needed lists and variables from the radar_dict configuration dictionary
comp_radar = config.radar_config_dict.get('comp_radar', False)
clean_radar = config.radar_config_dict.get('comp_radar', False)
calc_dualpol = config.radar_config_dict.get('calc_dualpol', False)
radar_name = config.radar_config_dict.get('radar_name', None)
radar_type = config.radar_config_dict.get('radar_type', None)
radar_dir = config.radar_config_dict.get('radar_dir', None)
radar_fname_pattern = config.radar_config_dict.get('radar_fname_pattern', None)
# Add the input filename tag to the pattern if needed
if input_tag:
    radar_fname_pattern = radar_fname_pattern.replace('.', '_{}.'.format(input_tag))
field_names = config.radar_config_dict.get('field_names', ['REF'])
if 'VEL' in field_names and dealias_vel and 'VEL_corrected' not in field_names:
    field_names.append("VEL_corrected")
if not calc_dualpol:
    field_names = ['REF']
if el_req_cl:
    el_req = el_req_cl
else:
    el_req = config.radar_config_dict.get('el_req', 0.5)
radar_start_timestamp = config.radar_config_dict.get('radar_start_timestamp', None)
radar_end_timestamp = config.radar_config_dict.get('radar_end_timestamp', None)
scatt_dir = config.radar_config_dict.get('scatt_dir', None)
wavelength = config.radar_config_dict.get('wavelength', 10.7)

In [ ]:
# Read in radar sweeps
sweeptime_list, radarsweep_list = read_sweeps(radname=radar_name, radardir=radar_dir,
                                              radstarttime=radar_start_timestamp, radstoptime=radar_end_timestamp,
                                              fieldnames=field_names, el_req=el_req)

In [ ]:
# First, we need to grid the input radar data.
radar_grid_list = []
for i, radarsweep in enumerate(radarsweep_list):
    print(f"Gridding radarsweep {i} with {radarsweep.nsweeps} sweeps")
    radar_grid = pyart.map.grid_ppi_sweeps(radarsweep)
    radar_grid_list.append(radar_grid)


radar_grid = xr.concat(radar_grid_list, dim='time')

In [ ]:
radar_grid_pyart = radar_grid

In [ ]:
dx = radar_grid_pyart['x'][1] - radar_grid_pyart['x'][0]
dy = radar_grid_pyart['y'][1] - radar_grid_pyart['y'][0]
print('dx =', dx.values, 'm')
print('dy =', dy.values, 'm')

In [ ]:
# We need to do some adjustments to the radar_grid_pyart xarray Dataset to make it compatible with tobac
# First, there is a time coordinate but no corresponding time dimension. We need to add a time
# dimension and associate it with the time coordinate. This is because tobac expects a time
# dimension in the input xarray Dataset.
# radar_grid_pyart = radar_grid_pyart.expand_dims(dim={'time': [sweeptime_list[0]]}, axis=0)
# Next, we need to rename the "elevation" coordinate to "z" because tobac expects a vertical
# coordinate named "z" However, there is already a "z" coordinate without a corresponding "z"
# dimension, so we need to drop the existing "z" coordinate before renaming "elevation" to "z"
# radar_grid_pyart = radar_grid_pyart.drop('z').rename({'elevation': 'z'})
# Change units of "z" coordinate to meters.
# radar_grid_pyart['z'].attrs["units"] = 'm'
# Actually just drop the z coordinate entirely since it's not needed for 2D segmentation and tracking and is causing some issues with tobac
radar_grid_pyart = radar_grid_pyart.squeeze() # Removes singular "z" dimension
radar_grid_pyart = radar_grid_pyart.drop('z')
radar_grid_pyart = radar_grid_pyart.drop('elevation')
# Need to drop z dimension too since we dropped the z coordinate
# radar_grid_pyart = radar_grid_pyart.drop_dims('z')
# Change standard names attributes of lat and lon coordinates to match tobac's expected coordinate names
radar_grid_pyart['lat'].attrs["standard_name"] = 'latitude'
radar_grid_pyart['lon'].attrs["standard_name"] = 'longitude'
# Change standard name attribute of the x and y coordinates to "projection_x_coordinate" and "projection_y_coordinate" to match tobac's expected coordinate names
radar_grid_pyart['x'].attrs["standard_name"] = 'projection_x_coordinate'
radar_grid_pyart['y'].attrs["standard_name"] = 'projection_y_coordinate'
# Add units attributes to the x and y coordinates to match tobac's expected coordinate attributes
radar_grid_pyart['x'].attrs["units"] = 'm'
radar_grid_pyart['y'].attrs["units"] = 'm'
# Finally, drop any other coordinates that are not needed and may be causing issues with tobac,
# like "origin_latitude", "origin_longitude", "radar_latitude", and "radar_longitude"
radar_grid_pyart = radar_grid_pyart.drop_vars(['origin_latitude', 'origin_longitude',
                                               'radar_latitude', 'radar_longitude'])

In [ ]:
radar_grid_pyart

In [ ]:
# Determine temporal and spatial sampling of the input data:

dxy, dt = tobac.get_spacings(radar_grid_pyart['REF'])
print(dxy, dt)

In [ ]:
reftime = "2025-06-07T00:17:56"
fig, ax = plt.subplots(figsize=[8, 8])
radar_grid_pyart['REF'].sel(time=reftime).plot(ax=ax)

In [ ]:
feature_detection_params = dict()
feature_detection_params["threshold"] = [30, 40, 50]
feature_detection_params["target"] = "maximum"
feature_detection_params["position_threshold"] = "weighted_diff"
feature_detection_params["n_erosion_threshold"] = 2
feature_detection_params["sigma_threshold"] = 1
feature_detection_params["n_min_threshold"] = 4

In [ ]:
# Perform feature detection:
savedir = Path('/Users/dawson29/Projects/pyPIPS/notebooks/ICECHIP_PIPS_model_comp/test_output')
if not savedir.exists():
    savedir.mkdir(parents=True, exist_ok=True)
print('starting feature detection')
radar_features = tobac.feature_detection.feature_detection_multithreshold(
    radar_grid_pyart['REF'], 0, **feature_detection_params
)
radar_features.to_hdf(savedir / 'Features.h5', 'table')
print('feature detection performed and saved')

In [ ]:
radar_features

In [ ]:
def get_nearest_row_by_time(features_df, target_time_str):
    # 1. Calculate the absolute difference between the target time and the times in the DataFrame
    datetimes = pd.to_datetime(features_df["timestr"])
    time_diffs = (datetimes - pd.to_datetime(target_time_str)).abs()

    # 2. Find the index of the minimum difference
    nearest_row_index = time_diffs.idxmin()

    # 3. Select the entire row using .loc[]
    nearest_row = features_df.loc[nearest_row_index]

    return nearest_row_index, nearest_row

def get_nearest_frame_by_time(features_df, target_time_str):
    nearest_row_index, nearest_row = get_nearest_row_by_time(features_df, target_time_str)
    nearest_frame = nearest_row["frame"]
    return nearest_frame

In [ ]:
radar_grid_pyart['REF'].where(radar_grid_pyart['REF'] > 0).sel(time=reftime).plot(vmin=0, vmax=65,
                                                                                  cmap="Spectral_r",
                                                                                  figsize=(8,8))

nearest_frame = get_nearest_frame_by_time(radar_features, reftime)

# Get all rows in the features DataFrame that correspond to the nearest frame
nearest_frame_rows = radar_features[radar_features["frame"] == nearest_frame]

plt.scatter(
    nearest_frame_rows["x"],
    nearest_frame_rows["y"],
    70,
    color="k",
)

In [ ]:
parameters_segmentation = dict()
parameters_segmentation["method"] = "watershed"
parameters_segmentation["threshold"] = 35
parameters_segmentation["target"] = "maximum"
# parameters_segmentation["seed_3D_flag"] = "box"
# parameters_segmentation["seed_3D_size"] = 5

In [ ]:
# Perform segmentation and save results:
print('Starting segmentation.')
segmentation_mask, segmentation_features = tobac.segmentation.segmentation(
    radar_features, radar_grid_pyart['REF'], dxy=dxy, **parameters_segmentation
)
print('segmentation performed, start saving results to files')
segmentation_mask.to_netcdf(savedir / 'Mask_Segmentation_rad.nc',
                            encoding={"segmentation_mask":{"zlib":True, "complevel":4}})
segmentation_features.to_hdf(savedir / 'Features_rad.h5', 'table')
print('segmentation performed and saved')


In [ ]:
segmentation_mask

In [ ]:
fig = plt.figure(figsize=[10, 8])
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
# ax.set_extent([-101.7, -100.7, 39, 40], crs=ccrs.PlateCarree())

contoured = ax.contourf(
    radar_grid_pyart['REF']["lon"],
    radar_grid_pyart['REF']["lat"],
    radar_grid_pyart['REF'][nearest_frame],
    transform=ccrs.PlateCarree(),
    cmap="Spectral_r", vmin=0, vmax=65
)
# plt.xlim(-101.7, -100.7)
# plt.ylim(39.0, 40)
unique_seg = np.unique(segmentation_mask.isel(time=nearest_frame))
color_map = cmaps.plasma(np.linspace(0, 1, len(unique_seg)))

for seg_num, color in zip(unique_seg, color_map):
    # if seg_num == 0 or seg_num == -1:
    #    continue
    curr_seg = (segmentation_mask == seg_num).astype(int)
    # print(curr_seg)
    ax.contour(
        segmentation_mask["lon"],
        segmentation_mask["lat"],
        curr_seg.isel(time=nearest_frame),
        colors=[
            color,
        ],
        levels=[
            0.9,1
        ],
        linewidths=3,
    )
    curr_feat = nearest_frame_rows[nearest_frame_rows["feature"] == seg_num]
    plt.scatter(
        curr_feat["lon"],
        curr_feat["lat"],
        70,
        transform=ccrs.PlateCarree(),
        color=color,
        edgecolors='black',
        zorder=10
    )

cb = plt.colorbar(contoured)
cb.set_label("Reflectivity", size=14)
cb.ax.tick_params(labelsize=14)

In [ ]:
# Keyword arguments for linking step:
parameters_linking={}
parameters_linking['method_linking']='predict'
# parameters_linking['adaptive_stop']=0.2
# parameters_linking['adaptive_step']=0.95
# parameters_linking['extrapolate']=0
# parameters_linking['order']=1
# parameters_linking['subnetwork_size']=100
# parameters_linking['memory']=0
# parameters_linking['time_cell_min']=5*60
# parameters_linking['method_linking']='predict'
parameters_linking['v_max'] = 30

In [ ]:
# Perform linking and save results:
Track = tobac.linking_trackpy(radar_features, radar_grid_pyart['REF'], dt=dt, dxy=dxy,
                              **parameters_linking)
Track.to_hdf(savedir / 'Track.h5', 'table')

In [ ]:
# Plot map with all individual tracks:
import cartopy.crs as ccrs
fig_map, ax_map = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})
ax_map = tobac.map_tracks(Track, axes=ax_map)

In [ ]:
# Set extent for maps plotted in the following cells ( in the form [lon_min,lon_max,lat_min,lat_max])
axis_extent=[-103.5, -100.5, 33.0, 34.5]

In [ ]:
animation_tobac = tobac.animation_mask_field(
    track=Track, features=radar_features, field=radar_grid_pyart['REF'], mask=segmentation_mask,
    axis_extent=axis_extent,
    vmin=0, vmax=65, extend='both', cmap='Spectral_r',
    interval=500, figsize=(8, 8),
    plot_outline=True, plot_marker=True, marker_track='x', plot_number=True, plot_features=True
)

In [ ]:
# Display animation:
from IPython.display import HTML, Image, display
HTML(animation_tobac.to_html5_video())

In [ ]:
Track

In [ ]:
# Based on the animation, the storm we want is identifed as cell #11
cell_11 = Track[Track['cell'] == 11]
cell_11

In [ ]:
# Plot the track of cell 11 on a map:
fig_cell_11, ax_cell_11 = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.PlateCarree()})
ax_cell_11 = tobac.map_tracks(cell_11, axes=ax_cell_11)
ax_cell_11.set_extent(axis_extent, crs=ccrs.PlateCarree())

In [ ]:
# Compute the storm motion vector for cell 11 using tobac's built in function:
storm_motion_cell_11 = tobac.analysis.cell_analysis.calculate_velocity(cell_11,
                                                                       return_components=True)

In [ ]:
# Compute the average storm motion vector for all times within 15 minutes on either side of the
# reference time:
time_window = timedelta(minutes=15)
start_time_window = pd.to_datetime(reftime) - time_window
end_time_window = pd.to_datetime(reftime) + time_window
# Get all rows in the cell 11 track that are within the time window
cell_11_time_window = cell_11[(pd.to_datetime(cell_11['timestr']) >= start_time_window) & (pd.to_datetime(cell_11['timestr']) <= end_time_window)]
# Now simply compute the average u and v components of the storm motion vector using the first and last rows in the time window
u_avg = (cell_11_time_window.iloc[-1]['x'] - cell_11_time_window.iloc[0]['x']) / (pd.to_datetime(cell_11_time_window.iloc[-1]['timestr']) - pd.to_datetime(cell_11_time_window.iloc[0]['timestr'])).total_seconds()
v_avg = (cell_11_time_window.iloc[-1]['y'] - cell_11_time_window.iloc[0]['y']) / (pd.to_datetime(cell_11_time_window.iloc[-1]['timestr']) - pd.to_datetime(cell_11_time_window.iloc[0]['timestr'])).total_seconds()
print("Average storm motion vector for cell 11 within +/- 15 minutes of reference time (u_avg, v_avg) in m/s:", (u_avg, v_avg))